# Figures 3 and 4: high-dimensional covariatesSweeps the covariate dimension `p` from 3 to 1,000 with the sample size heldfixed, under both a linear and a nonlinear data-generating process, and writesthe eight panels used by `fig:highdim_linear` and `fig:highdim_nonlinear`.| Figure | DGP | Files ||---|---|---|| 3 | linear | `amse_c-index_plots.png`, `train_test_cindex.png`, `train_test_amse.png`, `loss_trajectorie.png` || 4 | nonlinear | the same four names with a `1` suffix |The unsuffixed and `1`-suffixed names are fixed by the `\includegraphics`calls in the manuscript. They are easy to collide with, so nothing else in therepository writes them.Only the first three covariates carry signal; the remaining `p - 3` are noise.The quantity of interest is the **train-test gap**, which widens with `p`because the network has more noise dimensions to fit. Both curves are reported,not test alone.

In [ ]:
import os, syssys.path.insert(0, os.path.abspath(".."))   # repository root, so `rnn_agt` importsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport rnn_agtfrom rnn_agt import data as Dfrom rnn_agt.seeds import make_seedsfrom rnn_agt.train import TrainConfig, train_model, predictfrom rnn_agt.metrics import evaluateprint("rnn_agt", rnn_agt.__version__)

In [ ]:
import osP_VALUES = [3, 10, 30, 50, 100, 300, 500, 1000]N_TRAIN, N_TEST, CENSORING, EPOCHS = 1000, 2000, 0.25, 30# Figure 3 uses the linear signal, Figure 4 the richer nonlinear one. Both are# carried by the first three covariates only; the remaining p-3 are noise.#   linear:            z1 - 6 z2 + 4 z3#   highdim_nonlinear: 3 z1 - 6 z2 + 4 z3 + 2 z1 z2 - 3 z2^2 + sin(z3)DGPS = {"linear": "", "highdim_nonlinear": "1"}OUTDIR = "manuscript/images" if os.path.isdir("manuscript/images") else "results"os.makedirs(OUTDIR, exist_ok=True)print("writing figures to", OUTDIR)

In [ ]:
results = {}for mf, suffix in DGPS.items():    rows, curves = [], {}    for p_dim in P_VALUES:        seeds = make_seeds(31 + p_dim)        rng = seeds.data()        tau = D.calibrate_tau(N_TRAIN, D.MEAN_FUNCTIONS[mf], "normal", rng,                              CENSORING, D.DEPENDENCE_SPECS["ar1"])        tr = D.make_dataset(N_TRAIN, mf, "normal", rng,                            dependence="ar1", tau=tau, p=p_dim)        te = D.make_dataset(N_TEST, mf, "normal", rng,                            dependence="ar1", tau=tau, p=p_dim)        cfg = TrainConfig(model="rnn_agt", epochs=EPOCHS, pair_sample_s=30,                          hidden_dim=64, gru_layers=2, lr=3e-4)        res = train_model(tr, te, p_dim, cfg, make_seeds(11))        rows.append({"p": p_dim, **res.metrics, "params": res.n_params})        curves[p_dim] = res.epoch_losses        print(f"{mf:12s} p={p_dim:5d}  test C={res.metrics['test_cindex']:.3f}  "              f"test AMSE={res.metrics['test_amse']:.2f}", flush=True)    results[mf] = (pd.DataFrame(rows), curves)pd.concat([df.assign(dgp=mf) for mf, (df, _) in results.items()]).round(3)

In [ ]:
for mf, suffix in DGPS.items():    df, curves = results[mf]    # (a) train vs test C-index    fig, ax = plt.subplots(figsize=(4.6, 3.4))    ax.plot(df.p, df.train_cindex, marker="o", label="train")    ax.plot(df.p, df.test_cindex, marker="s", label="test")    ax.axhline(0.5, ls="--", c="grey", lw=1)    ax.set_xscale("log"); ax.set_xlabel("covariate dimension $p$")    ax.set_ylabel("IPCW C-index"); ax.legend(); ax.grid(alpha=.3)    fig.tight_layout()    fig.savefig(os.path.join(OUTDIR, f"train_test_cindex{suffix}.png"), dpi=200)    plt.close(fig)    # (b) train vs test AMSE    fig, ax = plt.subplots(figsize=(4.6, 3.4))    ax.plot(df.p, df.train_amse, marker="o", label="train")    ax.plot(df.p, df.test_amse, marker="s", label="test")    ax.set_xscale("log"); ax.set_xlabel("covariate dimension $p$")    ax.set_ylabel("AMSE"); ax.legend(); ax.grid(alpha=.3)    fig.tight_layout()    fig.savefig(os.path.join(OUTDIR, f"train_test_amse{suffix}.png"), dpi=200)    plt.close(fig)    # (c) AMSE against C-index    fig, ax = plt.subplots(figsize=(4.6, 3.4))    sc = ax.scatter(df.test_cindex, df.test_amse, c=np.log10(df.p),                    cmap="viridis", s=70)    for _, r in df.iterrows():        ax.annotate(f"{int(r.p)}", (r.test_cindex, r.test_amse),                    fontsize=7, xytext=(3, 3), textcoords="offset points")    ax.set_xlabel("test IPCW C-index"); ax.set_ylabel("test AMSE")    ax.grid(alpha=.3)    fig.colorbar(sc, ax=ax, label="$\\log_{10} p$")    fig.tight_layout()    fig.savefig(os.path.join(OUTDIR, f"amse_c-index_plots{suffix}.png"), dpi=200)    plt.close(fig)    # (d) loss trajectories    fig, ax = plt.subplots(figsize=(4.6, 3.4))    for p_dim in P_VALUES:        ax.plot(range(1, len(curves[p_dim]) + 1), curves[p_dim],                lw=1.2, label=f"p={p_dim}")    ax.set_xlabel("epoch"); ax.set_ylabel("mini-batch Gehan-WRS loss")    ax.legend(fontsize=6, ncol=2); ax.grid(alpha=.3)    fig.tight_layout()    fig.savefig(os.path.join(OUTDIR, f"loss_trajectorie{suffix}.png"), dpi=200)    plt.close(fig)    print(f"{mf}: wrote 4 panels with suffix {suffix!r}")print("\n8 panels written for fig:highdim_linear and fig:highdim_nonlinear.")print("AMSE is reported with the intercept fixed, so the widening train-test")print("gap reflects generalization rather than drift in the predictor level.")